# Apache Parquet - JavaScript

All 7 JavaScript examples from [docs/parquet.md](https://platob.github.io/yggdryl/parquet/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install yggdryl
```

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { IOBase } = require('yggdryl')

const table = new arrow.Table({
  id: arrow.vectorFromArray([1n, 2n, 3n], new arrow.Int64()),
  symbol: arrow.vectorFromArray(['AAPL', null, 'MSFT'], new arrow.Utf8()),
})

// The name says Parquet, so no call names an encoding.
const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const handle = new IOBase(path.join(root, 'trades.parquet'))
handle.writeArrowBatchReader(table)

// Reading streams: one batch at a time, never one materialized table.
assert.equal(handle.readArrowBatchReader().toTable().numRows, 3)
assert.deepEqual([...handle.readBytes().subarray(0, 4)], [...Buffer.from('PAR1')])

fs.rmSync(root, { recursive: true, force: true })

## Column pushdown

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { Field, IOBase, fields } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const handle = new IOBase(path.join(root, 'trades.parquet'))
handle.writeArrowBatchReader(
  new arrow.Table({
    id: arrow.vectorFromArray([1n, 2n], new arrow.Int64()),
    symbol: arrow.vectorFromArray(['AAPL', 'MSFT'], new arrow.Utf8()),
    price: arrow.vectorFromArray([1.5, 2.5], new arrow.Float64()),
    venue: arrow.vectorFromArray(['XNAS', 'XNAS'], new arrow.Utf8()),
  }),
)

// Two of the four columns, declared as this read's schema.
const wanted = fields.struct(
  'row',
  [Field.from('id: int64'), Field.from('price: float64')],
  { nullable: false },
)

const options = handle.recordOptions().withSchema(wanted)
const projected = handle.readArrowBatchReader(options).toTable()
assert.equal(projected.numCols, 2)
assert.deepEqual(projected.schema.fields.map((child) => child.name), ['id', 'price'])

// The file is unchanged: it still stores all four.
assert.equal(handle.readArrowField().dataType.length, 4)

fs.rmSync(root, { recursive: true, force: true })

## Options

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const handle = new IOBase(path.join(root, 'trades.parquet'))

// Parquet's own settings and the shared ones are properties of one value.
const options = handle
  .recordOptions()
  .withMaxRowGroupSize(4_096)
  .withKeyValue('iceberg.schema-id', '7')
  .withBatchSize(256)
  .withRootName('trade')

assert.equal(options.maxRowGroupSize, 4_096)
assert.deepEqual(options.keyValueMetadata, [{ key: 'iceberg.schema-id', value: '7' }])
assert.equal(options.batchSize, 256)
assert.equal(options.rootName, 'trade')
assert.equal(options.safe, false)

const ids = Array.from({ length: 1_000 }, (_, index) => BigInt(index))
handle.writeArrowBatchReader(
  new arrow.Table({ id: arrow.vectorFromArray(ids, new arrow.Int64()) }),
  options,
)

// batchSize bounds the reader, so no batch holds all 1,000 rows.
const counts = [...handle.readArrowBatchReader(options)].map((batch) => batch.numRows)
assert.equal(counts.reduce((total, count) => total + count, 0), 1_000)
assert.ok(counts.every((count) => count <= 256), counts.join())

// The root name names the Field recovered from the footer.
assert.equal(handle.readArrowField(options).name, 'trade')

fs.rmSync(root, { recursive: true, force: true })

## Compression

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const ids = Array.from({ length: 4_000 }, (_, index) => BigInt(index))
const table = new arrow.Table({
  id: arrow.vectorFromArray(ids, new arrow.Int64()),
  symbol: arrow.vectorFromArray(ids.map(() => 'AAPL'), new arrow.Utf8()),
})

const sizes = []
for (const compression of ['uncompressed', 'snappy', 'zstd(1)']) {
  const handle = new IOBase(path.join(root, `trades-${compression}.parquet`))
  // One batch per read, so the comparison is not split by the default bound.
  const options = handle
    .recordOptions()
    .withCompression(compression)
    .withBatchSize(table.numRows)
  handle.writeArrowBatchReader(table, options)

  // Nothing on the read side names the compression: the footer records it.
  const read = handle.readArrowBatchReader(options).toTable()
  assert.equal(read.numRows, 4_000, compression)
  sizes.push(handle.size)
}

assert.ok(sizes[0] > sizes[1] && sizes[0] > sizes[2], sizes.join())

fs.rmSync(root, { recursive: true, force: true })

## Coded handles are rejected

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))

// The name declares gzip over the Parquet file.
const handle = new IOBase(path.join(root, 'trades.parquet.gz'))

assert.throws(
  () =>
    handle.writeArrowBatchReader(
      new arrow.Table({ id: arrow.vectorFromArray([1n], new arrow.Int64()) }),
    ),
  /parquet compresses/,
)

// Nothing was published.
assert.equal(handle.size, 0)

fs.rmSync(root, { recursive: true, force: true })

## Field identifiers

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { IOBase } = require('yggdryl')

// Arrow JS carries the identifiers the same way Arrow does anywhere else:
// as field metadata under the exact `PARQUET:field_id` key.
const rows = new arrow.Table({
  id: arrow.vectorFromArray([1n], new arrow.Int64()),
  symbol: arrow.vectorFromArray(['AAPL'], new arrow.Utf8()),
})
const schema = new arrow.Schema(
  rows.schema.fields.map(
    (child, index) =>
      new arrow.Field(
        child.name,
        child.type,
        child.nullable,
        new Map([['PARQUET:field_id', String(index + 1)]]),
      ),
  ),
)

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const handle = new IOBase(path.join(root, 'trades.parquet'))
handle.writeArrowBatchReader(new arrow.Table(schema, rows.batches[0].data))

// The ids went into the file, so the recovered Field answers by id rather
// than by position.
const recovered = handle.readArrowField()
assert.deepEqual([...recovered.dataType].map((child) => child.parquetFieldId), [1, 2])
assert.equal(recovered.dataType.at(0).get('PARQUET:field_id'), '1')

fs.rmSync(root, { recursive: true, force: true })

## The handle underneath

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))

// Nothing has been written, so there is nothing to read.
const empty = new IOBase(path.join(root, 'absent.parquet'))
assert.equal(empty.readArrowBatchReader().toTable().numRows, 0)

// An empty write still publishes a readable file with the schema in its footer.
const schema = new arrow.Schema([new arrow.Field('id', new arrow.Int64(), true)])
const handle = new IOBase(path.join(root, 'written.parquet'))
handle.writeArrowBatchReader(new arrow.Table(schema))
assert.ok(handle.size > 0)
assert.equal(handle.readArrowBatchReader().toTable().numRows, 0)
assert.equal(handle.readArrowField().dataType.length, 1)

fs.rmSync(root, { recursive: true, force: true })